# Lecture 2: Shooting the swing-up

**In this lecture.** We solve the boundary value problem derived in
Lecture 1 for Kelly's cart-pole swing-up. We start with the naive approach,
guessing the initial costate, and see why it fails. Then we try two remedies
that practitioners actually use: random multistart, and seeding the shooting
from a cheap direct (collocation) solution. Along the way we find that the
necessary conditions have *more than one* solution here, so "the" optimal
swing-up has a competitor. We finish by checking the result independently of
the package.

**Prerequisites.** Lecture 1: the canonical equations, the shooting residual
and the state transition matrix.

**After this lecture you can**

- set up and solve a single-shooting problem with the package;
- explain why indirect methods need good costate guesses, and get them from
  random search or from a direct method;
- tell a converged extremal from a verified optimum, and check a solution
  independently.

Background: Stanford AA203 lecture 5 ("indirect methods: practical aspects"),
MIT 16.323 lecture notes 6 and 8, and Kelly (2017), the source of the
problem and of the direct method used in section 4.

## 1. The problem, posed for the package

The same plant and boundary conditions as Lecture 1, with Kelly's cost
$J = \int_0^T u^2\,dt$ and Kelly's force limit $|u| \le 20$ N. The control
law is then the clipped one from Lecture 1, section 5.

The limit turns out to be inactive at the solution (we check that in
section 5), but it still matters, for the solver rather than the answer. With
no bound, a wild costate guess commands an enormous force, the cart-pole
spins up violently, and the integrator crawls through it. One failed start
can then take seconds instead of milliseconds. With the bound, every guess
integrates quickly. Kelly also limits the track to $|q_1| \le 2$ m. We leave
that out and check it at the end. (Lecture 3 tightens the force limit until it
binds, and Lecture 5 shortens the track.)

The constants are SymPy symbols, so they become **runtime parameters** of the
compiled kernels: one compilation serves every value.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import Box, OptimalControlProblem
from auto_optimal_control.shooting import ShootingError, SingleShooting

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
q1, q2, w1, w2, u = sp.symbols("q_1 q_2 qd_1 qd_2 u", real=True)
m1, m2, l, g, u_max, d, T = sp.symbols("m_1 m_2 l g u_max d T", positive=True)

den = m1 + m2 * (1 - sp.cos(q2) ** 2)
q1dd = (
    l * m2 * sp.sin(q2) * w2**2 + u + m2 * g * sp.cos(q2) * sp.sin(q2)
) / den
q2dd = -(
    l * m2 * sp.cos(q2) * sp.sin(q2) * w2**2
    + u * sp.cos(q2)
    + (m1 + m2) * g * sp.sin(q2)
) / (l * den)

problem = OptimalControlProblem(
    name="cart_pole",
    time=t,
    states=[q1, q2, w1, w2],
    controls=[u],
    params=[m1, m2, l, g, u_max, d, T],
    dynamics=[w1, w2, q1dd, q2dd],
    running_cost=u**2,
    initial_state=[0, 0, 0, 0],
    final_state={q1: d, q2: sp.pi, w1: 0, w2: 0},
    final_time=T,
    control_sets=[Box(u, -u_max, u_max)],
)

start = time.perf_counter()
kernels = build_kernels(derive(problem))
print(f"derived and compiled in {time.perf_counter() - start:.1f} s")
p = [1.0, 0.3, 0.5, 9.81, 20.0, 1.0, 2.0]  # m1, m2, l, g, u_max, d, T
kernels.param_names

`SingleShooting` wraps the kernels in the residual $R(v)$ of Lecture 1 and its
exact Jacobian, where $v$ collects the unknowns: the initial costate
$\lambda_0$ and the multipliers $\nu$ of the terminal conditions. `solve` hands
both to SciPy's Powell hybrid method. `guess_multipliers` takes a costate
guess and fills in the $\nu$ that best fit it. The transversality conditions
are linear in $\nu$, so only $\lambda_0$ has to be guessed.

In [ ]:
shoot = SingleShooting(kernels, p)
shoot

## 2. A first guess

With no information, the natural guess is $\lambda_0 = 0$: no idea which way
to push. Try it.

In [ ]:
sol0 = shoot.solve(shoot.guess_multipliers(np.zeros(4)))
J0 = kernels.propagate(0.0, sol0.z0, p[-1], p, cost=True)["cost"]
print(f"residual norm {sol0.residual_norm:.1e}, success: {sol0.success}")
print(f"J = {J0:.6f}")
print("lambda0 =", np.round(sol0.lam0, 4))

It converged, to a swing-up with $J \approx 57.94$. That is luck, not
design. At $\lambda_0 = 0$ the costate stays zero, the force is zero, and the
pendulum just hangs: the first Newton step has nothing to go on but the
Jacobian there, and it happens to point somewhere useful. Nothing guarantees
that for another problem, or even for this one with other parameters.

The underlying trouble is **sensitivity**. The cell below integrates from
a few random costates and reports how large the state transition matrix gets
by $t = T$, together with the condition number of the shooting Jacobian
$\partial x(T) / \partial \lambda_0$. A small change in $\lambda_0$ makes a
large change at the far end, and the Jacobian is far from well conditioned.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(4):
    lam0 = rng.uniform(-30, 30, 4)
    r = kernels.propagate(0.0, kernels.z0(lam0, p), p[-1], p, stm=True)
    Phi = r["stm"]
    print(
        f"|Phi(T)| = {np.linalg.norm(Phi):9.3g}   "
        f"cond(dx(T)/dlam0) = {np.linalg.cond(Phi[:4, 4:]):9.3g}"
    )

## 3. Remedy one: random multistart

The bluntest remedy is to try many guesses. Each `solve` is cheap, because the
kernels are compiled C integrated by a compiled integrator, so we can afford
hundreds. We draw $\lambda_0$ uniformly from a box, solve from each draw, and
keep every converged extremal, grouped by its cost.

In [ ]:
def solve_from(lam0):
    sol = shoot.solve(shoot.guess_multipliers(lam0))
    if not sol.success:
        return None
    cost = kernels.propagate(0.0, sol.z0, p[-1], p, cost=True)["cost"]
    return sol, cost


rng = np.random.default_rng(0)
found = {}
start = time.perf_counter()
n_tries, n_converged = 200, 0
for _ in range(n_tries):
    try:
        out = solve_from(rng.uniform(-60, 60, 4))
    except (ShootingError, RuntimeError, ValueError, np.linalg.LinAlgError):
        continue
    if out is not None:
        sol, J = out
        n_converged += 1
        key = round(J, 6)
        found.setdefault(key, [sol, 0])[1] += 1
elapsed = time.perf_counter() - start
print(f"{n_tries} starts in {elapsed:.1f} s, {n_converged} converged")
for J in sorted(found):
    print(f"J = {J:10.6f}   found {found[J][1]:3d} times")

Only about one start in eight converges, and every one that does lands on the
same extremal as the zero guess, $J \approx 57.94$. That says something about
the basins of attraction of Newton's method in costate space: they are small
and irregular, and a uniform box mostly misses them. It does *not* say that
this extremal is the only one. Section 4 finds another.

## 4. Remedy two: seed from a direct method

A **direct** method discretizes first and optimizes second. It turns the
problem into a finite nonlinear program over the states and controls at grid
points, and gives that to a general-purpose optimizer. It is robust to poor
guesses (a straight line will do) but only as accurate as its grid. An
**indirect** method is the reverse: accurate to integration tolerance, but
fragile. Using a direct solution to seed an indirect one combines the two
strengths (AA203 lecture 5; Betts, *Practical Methods for Optimal Control*).

The direct method below is Kelly's: Hermite–Simpson collocation on 40
intervals, with SciPy's SLSQP. The *defects* are the collocation equations,
the requirement that each interval's cubic interpolant satisfy the dynamics
at its ends and its midpoint. It needs a NumPy version of the plant.

In [ ]:
def plant(x, u, p):
    m1, m2, l, g = p[:4]
    q2, w2 = x[1], x[3]
    den = m1 + m2 * (1 - np.cos(q2) ** 2)
    return np.array(
        [
            x[2],
            w2,
            (
                l * m2 * np.sin(q2) * w2**2
                + u
                + m2 * g * np.cos(q2) * np.sin(q2)
            )
            / den,
            -(
                l * m2 * np.cos(q2) * np.sin(q2) * w2**2
                + u * np.cos(q2)
                + (m1 + m2) * g * np.sin(q2)
            )
            / (l * den),
        ]
    )


def hermite_simpson(p, N=40, x_max=2.0):
    from scipy.optimize import minimize

    d, T = p[5], p[6]
    h = T / N

    def split(w):
        return w[: 4 * (N + 1)].reshape(N + 1, 4), w[4 * (N + 1) :]

    def defects(w):
        X, U = split(w)
        out = []
        for i in range(N):
            fk, fk1 = plant(X[i], U[i], p), plant(X[i + 1], U[i + 1], p)
            xm = 0.5 * (X[i] + X[i + 1]) + h / 8 * (fk - fk1)
            um = 0.5 * (U[i] + U[i + 1])
            out.append(
                X[i + 1] - X[i] - h / 6 * (fk + 4 * plant(xm, um, p) + fk1)
            )
        out += [X[0], X[-1] - [d, np.pi, 0, 0]]
        return np.concatenate(out)

    def cost(w):
        _, U = split(w)
        um = 0.5 * (U[:-1] + U[1:])
        return h / 6 * np.sum(U[:-1] ** 2 + 4 * um**2 + U[1:] ** 2)

    tg = np.linspace(0, T, N + 1)
    # Straight-line guess: cart and pole move uniformly, at zero velocity.
    line = np.stack([d * tg / T, np.pi * tg / T, 0 * tg, 0 * tg], axis=1)
    w0 = np.concatenate([line.ravel(), np.zeros(N + 1)])
    bounds = [(-x_max, x_max), (None, None), (None, None), (None, None)] * (
        N + 1
    ) + [(-p[4], p[4])] * (N + 1)
    res = minimize(
        cost,
        w0,
        method="SLSQP",
        constraints={"type": "eq", "fun": defects},
        bounds=bounds,
        options={"maxiter": 500, "ftol": 1e-12},
    )
    X, U = split(res.x)
    return tg, X, U, cost(res.x)


start = time.perf_counter()
tg, X, U, J_direct = hermite_simpson(p)
elapsed = time.perf_counter() - start
print(f"direct solution: J = {J_direct:.5f} in {elapsed:.1f} s")

### Costates from a direct solution

The direct solution gives states and controls but no costate. We can recover
one. Along a *fixed* trajectory $(x(t), u(t))$ the costate equation
$\dot\lambda = -f_x^\top \lambda - L_x^\top$ is **linear** in $\lambda$ (here
$L_x = 0$), so $\lambda(t) = \Psi(t)\,\lambda_0$, where $\Psi$ is its
fundamental matrix. The control law of Lecture 1,
$u = -\tfrac12 f_u^\top \lambda$, then reads

$$
u(t_j) = -\tfrac12 f_u(t_j)^\top \Psi(t_j)\, \lambda_0
\qquad \text{at every sample time } t_j ,
$$

which is linear in $\lambda_0$: a least-squares problem with four unknowns and
as many equations as samples. (Nonlinear programming solvers also report
multipliers for the defect constraints, which approximate the costate
directly. The least-squares route needs nothing from the optimizer.)

In [ ]:
def costates_from_direct(tg, X, U, p):
    from scipy.integrate import solve_ivp
    from scipy.interpolate import CubicSpline

    Xs, Us = CubicSpline(tg, X), CubicSpline(tg, U)

    def jac(x, u, e=1e-7):
        fx = np.column_stack(
            [
                (plant(x + dx, u, p) - plant(x - dx, u, p)) / (2 * e)
                for dx in np.eye(4) * e
            ]
        )
        fu = (plant(x, u + e, p) - plant(x, u - e, p)) / (2 * e)
        return fx, fu

    def adjoint(s, Y):
        fx, _ = jac(Xs(s), float(Us(s)))
        return (-fx.T @ Y.reshape(4, 4)).ravel()

    Psi = solve_ivp(
        adjoint,
        (tg[0], tg[-1]),
        np.eye(4).ravel(),
        dense_output=True,
        rtol=1e-10,
        atol=1e-12,
    )
    samples = np.linspace(tg[0], tg[-1], 202)[1:-1]
    rows = [
        -0.5 * jac(Xs(s), float(Us(s)))[1] @ Psi.sol(s).reshape(4, 4)
        for s in samples
    ]
    return np.linalg.lstsq(np.array(rows), Us(samples), rcond=None)[0]


lam0_direct = costates_from_direct(tg, X, U, p)
sol = shoot.solve(shoot.guess_multipliers(lam0_direct))
J_polished = kernels.propagate(0.0, sol.z0, p[-1], p, cost=True)["cost"]
print("residual norm:", sol.residual_norm)
print(f"J direct   = {J_direct:.8f}  (Hermite-Simpson, N = 40)")
print(f"J polished = {J_polished:.8f}  (indirect, integration tolerance)")
print("published  = 58.80819      (Kelly's problem, high-order direct method)")

The polish converges to the extremal the direct method was near, which is the
one in the literature: Kelly's published cost, reproduced to the five figures
the published high-order direct method carries. The polish improves on our
own 40-interval direct solution in the fifth significant figure, which is that
grid's discretization error.

But this is **not** the extremal that multistart found. The necessary
conditions have at least two solutions here, and the one in the literature is
the more expensive. Kelly's direct method started from a straight line and
converged to a local minimum; the random search happened to find a better
one. Neither method promises a global optimum.

In [ ]:
J_best = min(found)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
for v, J, color in [
    (sol.v, J_polished, BLUE),
    (found[J_best][0].v, J_best, ORANGE),
]:
    tr = shoot.trajectory(v, n_points=401)
    axes[0].plot(
        tr["t"], tr["z"][:, 0], color=color, lw=2, label=f"J = {J:.4f}"
    )
    axes[1].plot(tr["t"], tr["z"][:, 1], color=color, lw=2)
    axes[2].plot(tr["t"], tr["u"][:, 0], color=color, lw=2)
axes[0].plot(tg, X[:, 0], "o", color=GRAY, ms=3, label="direct, N = 40")
axes[1].plot(tg, X[:, 1], "o", color=GRAY, ms=3)
axes[2].plot(tg, U, "o", color=GRAY, ms=3)
titles = ["cart position $q_1$ (m)", "pole angle $q_2$ (rad)", "force $u$ (N)"]
for ax, label in zip(axes, titles):
    ax.set_title(label, fontsize=10, loc="left")
    ax.set_xlabel("t (s)")
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

The two swing-ups are genuinely different. The literature extremal (blue)
pushes the cart forward at once, overshoots the target to 1.18 m and comes
back. The cheaper one (orange) first pulls the cart *backwards*, which kicks
the pole the other way, and never passes $q_1 = 1$. The grey dots are the
direct solution on its grid, which sits on the blue curve.

## 5. Checking the answer

A converged shooting solve says $R(v) \approx 0$ in the package's own
equations. An independent check uses none of them: take the extremal's
control history $u(t)$, drive Kelly's equations of motion with SciPy's
integrator, and see whether the cart arrives at the target with the claimed
cost. We also check two properties the theory predicts:

- $H$ is constant along the extremal, since the problem is autonomous;
- the force limit is inactive ($|u| < 20$ N throughout, so the clipped law
  never clips), and the track limit $|q_1| \le 2$ m that we left out holds
  anyway.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline


def check(v, label):
    tr = shoot.trajectory(v, n_points=4001)
    u_of_t = CubicSpline(tr["t"], tr["u"][:, 0])

    def f(s, y):
        us = float(u_of_t(s))
        return np.append(plant(y[:4], us, p), us**2)

    sim = solve_ivp(
        f, (0.0, p[-1]), np.zeros(5), method="DOP853", rtol=1e-12, atol=1e-12
    )
    end_error = np.abs(sim.y[:4, -1] - [p[5], np.pi, 0, 0]).max()
    H_spread = np.ptp(tr["H"]) / abs(tr["H"].mean())
    u_peak = np.abs(tr["u"]).max()
    q1_peak = np.abs(tr["z"][:, 0]).max()
    J_sim = sim.y[4, -1]
    print(f"{label}:")
    print(f"    simulated J = {J_sim:.8f}, end-state error {end_error:.1e}")
    print(f"    H varies by {H_spread:.0e} (relative)")
    print(f"    max|u| = {u_peak:.2f} N, max|q1| = {q1_peak:.2f} m")


check(sol.v, "literature extremal")
check(found[J_best][0].v, "cheapest extremal")

Both extremals check out. The simulated cost matches to eight figures, the
cart arrives to within $10^{-7}$ (the spline of $u(t)$ limits this), $H$ is
constant to better than $10^{-7}$ of its value, and both stay inside Kelly's
bounds. The force
peaks well below 20 N, which is why Lecture 3 has to tighten the limit to see
it bind.

## Summary

- Single shooting solves the boundary value problem by root-finding on the
  initial costate. It converges fast and accurately from a good guess, but
  costates have no physical intuition and the map $\lambda_0 \mapsto x(T)$ is
  sensitive, so good guesses are the hard part.
- Two remedies: many random starts, made affordable by compiled kernels, and
  a direct solution, whose costate can be recovered by least squares from
  the linear costate equation along its trajectory.
- The necessary conditions can have several solutions. Ranking them needs the
  cost, and proving global optimality needs more than first-order
  conditions. Neither method guarantees it.

## Exercises

1. Change the box in section 3 from $[-60, 60]^4$ to $[-10, 10]^4$ and then
   $[-200, 200]^4$. How does the fraction of converged starts change? Does
   either box find the literature extremal? (Hint: look at its $\lambda_0$.)
2. Run `hermite_simpson` with $N = 10, 20, 80$ and polish each. Plot
   $|J_\text{direct} - J_\text{polished}|$ against $N$ on log axes. What order
   of convergence do you see? (Hermite–Simpson is fourth order in the state;
   Kelly 2017, section 4.)
3. Seed SLSQP from the cheapest extremal's trajectory instead of a straight
   line. Does the direct method then find the cheaper solution?

## Further reading

- M. Kelly, "An Introduction to Trajectory Optimization: How to Do Your Own
  Direct Collocation", SIAM Review 59(4), 2017, sections 4 and 6.
- Stanford AA203, lecture 5: indirect methods in practice.
- J. T. Betts, *Practical Methods for Optimal Control Using Nonlinear
  Programming* (SIAM, 2010), chapter 4, on direct methods and costate
  estimates.
- MIT 16.323, lecture notes 8: properties of the optimal solution.